# FrameWeave v6 multi-seed — Qwen2.5-VL-7B train

Run All trains **GlobalTimeFix**, **HybridNoGlobal**, and **Hybrid** on seeds **1, 100, 2027**.

The PerceptionTest split stays on seed **42**. `--seed` only changes initialization and the per-epoch frame/option schedule. Each seed writes `outputs_reljoin_v6_qwen_multiseed/seed_{N}` and is mirrored to `MyDrive/reljoin_v6_qwen_multiseed_outputs`.

GlobalTimeFix turns the local branch off. HybridNoGlobal turns the global branch off; its local branch still receives global summaries. These are the Qwen flags, not the Ovis ablation rewrites. A variant that already has `checkpoints/best_external_val/{variant}.pt` is skipped. An interrupted variant trains again from epoch 1.

Run All clones [FrameWeave](https://github.com/nqMinh-GR/FrameWeave.git) into `/content/FrameWeave` and links `reljoin/` plus the runners into `/content`. Do not upload `.py` files. Unit tests read the NExT-GQA runner even though this notebook does not run NExT-GQA. Do not reuse `outputs_reljoin_v6_qwen`.


## 1. Configuration


In [ ]:
# 1. Configuration — Run All trains three Qwen variants × seeds 1, 100, 2027
from __future__ import annotations
from pathlib import Path

SEEDS = [1, 100, 2027]
SPLIT_SEED = 42
# Qwen flags, not the Ovis ablation rewrites:
# GlobalTimeFix = local branch off; HybridNoGlobal = global branch off;
# Hybrid = full hybrid. Local still sees global summaries in HybridNoGlobal.
VARIANTS = ["GlobalTimeFix", "HybridNoGlobal", "Hybrid"]
STAGE = "joint"

ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

OUT_ROOT = ROOT / "outputs_reljoin_v6_qwen_multiseed"
PERSIST = Path("/content/drive/MyDrive/reljoin_v6_qwen_multiseed_outputs")
MODEL_DIR = ROOT / "checkpoints" / "qwen25vl_7b"
PYTHON = ROOT / "qwen_env" / "bin" / "python"

print({
    "SEEDS": SEEDS,
    "SPLIT_SEED": SPLIT_SEED,
    "VARIANTS": VARIANTS,
    "STAGE": STAGE,
    "OUT_ROOT": str(OUT_ROOT),
    "PERSIST": str(PERSIST),
    "backbone": "Qwen/Qwen2.5-VL-7B-Instruct",
})


## 2. Environment (`qwen_env`)


In [ ]:
# 2. Bootstrap isolated qwen_env (Python 3.11 + torch 2.4.0 + transformers 4.51.3)
from __future__ import annotations
import os, subprocess
from pathlib import Path

assert Path("/content").exists(), "This notebook targets Google Colab (/content)."
ROOT = Path("/content")
VENV = ROOT / "qwen_env"
VENV_BIN = VENV / "bin"
PYTHON = VENV_BIN / "python"
PIP = VENV_BIN / "pip"
LOG = ROOT / "outputs_reljoin_v6_qwen_multiseed" / "logs" / "cell4_env.log"
LOG.parent.mkdir(parents=True, exist_ok=True)
LOG.write_text("", encoding="utf-8")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd, check=True, env=None):
    line = "$ " + " ".join(map(str, cmd))
    print(line, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        proc = subprocess.run(
            list(map(str, cmd)),
            check=False,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            env=env,
        )
        f.write(proc.stdout or "")
        f.write(f"\nexit={proc.returncode}\n\n")
    print(proc.stdout or "", end="")
    if check and proc.returncode != 0:
        print("LOG:", LOG)
        print((proc.stdout or "")[-4000:])
        raise RuntimeError(f"command failed ({proc.returncode}). See {LOG}")
    return proc

sh(["apt-get", "update", "-qq"], check=False)
sh(["apt-get", "install", "-y", "-qq", "python3.11", "python3.11-venv", "python3.11-dev", "ffmpeg"], check=False)
py311 = Path("/usr/bin/python3.11")
assert py311.exists(), "Không tìm thấy python3.11 — kiểm tra apt install."

need_recreate = True
if PYTHON.exists():
    ver = subprocess.check_output([str(PYTHON), "-V"], text=True).strip()
    print("Existing venv:", ver)
    need_recreate = "3.11" not in ver
if need_recreate:
    if VENV.exists():
        sh(["rm", "-rf", str(VENV)])
    sh([str(py311), "-m", "venv", "--copies", str(VENV)])
if not PIP.exists():
    sh([str(PYTHON), "-m", "ensurepip", "--upgrade"], check=False)
if not PIP.exists():
    get_pip = Path("/tmp/get-pip.py")
    sh(["curl", "-sSL", "https://bootstrap.pypa.io/get-pip.py", "-o", str(get_pip)])
    sh([str(PYTHON), str(get_pip)])
sh([str(PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools<70", "wheel"])
print("Python in venv:", subprocess.check_output([str(PYTHON), "-V"], text=True).strip())
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
    "torch==2.4.0", "torchvision==0.19.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])
sh([
    str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "--upgrade-strategy", "only-if-needed",
    "transformers==4.51.3", "numpy==1.25.0", "pillow==10.3.0", "moviepy==1.0.3",
    "huggingface_hub", "hf-transfer", "pyarrow==25.0.1", "datasets", "accelerate", "einops", "sentencepiece",
    "opencv-python-headless", "pandas", "scipy", "tqdm", "ninja", "packaging", "qwen-vl-utils",
])
print("Installing qwen-vl-utils; flash-attn is optional (SDPA is enough)...")
fa_env = os.environ.copy()
fa_env["MAX_JOBS"] = "4"
sh(
    [str(PYTHON), "-m", "pip", "install", "flash-attn==2.7.0.post2", "--no-build-isolation"],
    env=fa_env,
    check=False,
)
check = subprocess.check_output([
    str(PYTHON), "-c",
    "import torch, transformers, qwen_vl_utils; "
    "print(torch.__version__); print(transformers.__version__); print(qwen_vl_utils.__file__); "
    "print(torch.__file__); print(transformers.__file__); "
    "import importlib.util, numpy, pyarrow; print('numpy', numpy.__version__); print('pyarrow', pyarrow.__version__); assert numpy.__version__.startswith('1.'), numpy.__version__; assert pyarrow.__version__ == '25.0.1', pyarrow.__version__; print('flash_attn', bool(importlib.util.find_spec('flash_attn')))"
], text=True)
print(check)
for line in check.splitlines():
    if "/site-packages/" in line:
        assert "/content/qwen_env/" in line, f"Package leaked outside venv: {line}"
print("qwen_env ready:", PYTHON)
print("LOG:", LOG)


In [ ]:
# 2b. Extra pip (same slot as Video-LLaVA cell 5). Writes a log so a failure is inspectable.
from __future__ import annotations
import os, subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "qwen_env" / "bin" / "python"
LOG_DIR = ROOT / "outputs_reljoin_v6_qwen_multiseed" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / "cell5_pip.log"

if not PYTHON.exists():
    msg = (
        f"Missing {PYTHON}. Run the qwen_env bootstrap cell first "
        "(section 2), then re-run this cell.\n"
    )
    LOG.write_text(msg, encoding="utf-8")
    print(msg)
    raise FileNotFoundError(msg)

cmd = [
    str(PYTHON), "-m", "pip", "install",
    "--disable-pip-version-check",
    "--upgrade-strategy", "only-if-needed",
    "scikit-learn==1.3.2",
    "matplotlib==3.8.4",
    "numpy==1.25.0",
]
print("$", " ".join(cmd))
proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
log_text = f"$ {' '.join(cmd)}\nexit={proc.returncode}\n\n{proc.stdout or ''}"
LOG.write_text(log_text, encoding="utf-8")
print(proc.stdout or "")
print("LOG:", LOG)

persist = Path(os.environ.get("RELJOIN_PERSIST", "/content/drive/MyDrive/reljoin_v6_qwen_multiseed_outputs"))
if persist.parent.exists():
    try:
        persist.mkdir(parents=True, exist_ok=True)
        dest = persist / "logs" / LOG.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(log_text, encoding="utf-8")
        print("Copied log →", dest)
    except OSError as exc:
        print("Persist copy skipped:", exc)

if proc.returncode != 0:
    raise RuntimeError(f"cell 5 pip failed ({proc.returncode}). See {LOG}")
print("cell 5 pip OK")


## 3. Drive restore


In [ ]:
# 3. Mount Drive and restore finished seed folders
from __future__ import annotations
import os
import shutil
from pathlib import Path

ROOT = Path("/content")
OUT_ROOT = ROOT / "outputs_reljoin_v6_qwen_multiseed"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
try:
    SEEDS
except NameError:
    SEEDS = [1, 100, 2027]

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PERSIST = Path("/content/drive/MyDrive/reljoin_v6_qwen_multiseed_outputs")
except Exception as exc:
    print("Drive mount skipped:", exc)
    PERSIST = OUT_ROOT / "persist_local"
PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["RELJOIN_ROOT"] = str(ROOT)
print({"PERSIST": str(PERSIST), "OUT_ROOT": str(OUT_ROOT)})

for seed in SEEDS:
    src = PERSIST / f"seed_{seed}"
    dst = OUT_ROOT / f"seed_{seed}"
    if not src.exists() or src.resolve() == dst.resolve():
        print(f"seed {seed}: no Drive copy")
        continue
    print(f"restore {src} -> {dst}")
    shutil.copytree(src, dst, dirs_exist_ok=True)


## 4. Clone source and verify


In [ ]:
# Fetch FrameWeave source. Run All clones the repo; do not upload .py files.
from __future__ import annotations

import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nqMinh-GR/FrameWeave.git"
REPO_DIR = Path("/content/FrameWeave")
ROOT = Path("/content")
assert ROOT.exists(), "This notebook targets Google Colab (/content)."

if not (REPO_DIR / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])


def link(src: Path, dest: Path) -> None:
    src = src.resolve()
    if dest.is_symlink():
        if dest.resolve() != src:
            dest.unlink()
            dest.symlink_to(src)
        return
    if dest.exists():
        raise RuntimeError(
            f"{dest} already exists and is not the repo link. Restart the runtime, then Run All."
        )
    dest.symlink_to(src)


CODE = REPO_DIR / "qwen"
link(CODE / "reljoin", ROOT / "reljoin")
for script in sorted(CODE.glob("run_*.py")):
    link(script, ROOT / script.name)
print("FrameWeave source linked from", CODE)


In [ ]:
# 4. Verify FrameWeave sources
from pathlib import Path
ROOT = Path("/content")
required = [
    ROOT / "run_reljoin.py",
    ROOT / "run_herbench_oracle_reljoin.py",
    ROOT / "run_nextgqa_reljoin.py",
    ROOT / "reljoin" / "__init__.py",
    ROOT / "reljoin" / "config.py",
    ROOT / "reljoin" / "reljoin_modules.py",
    ROOT / "reljoin" / "qwen_hooks.py",
    ROOT / "reljoin" / "qwen_generate.py",
    ROOT / "reljoin" / "unit_tests.py",
    ROOT / "reljoin" / "train.py",
    ROOT / "reljoin" / "data_perception.py",
    ROOT / "reljoin" / "frame_io.py",
    ROOT / "reljoin" / "losses.py",
    ROOT / "reljoin" / "diagnostics.py",
    ROOT / "reljoin" / "herbench_oracle.py",
    ROOT / "reljoin" / "stats.py",
    ROOT / "reljoin" / "nextgqa.py",
    ROOT / "reljoin" / "nextgqa_frames.py",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing files — clone or symlink failed:\n" + "\n".join(missing)
print("All FrameWeave Hybrid sources present.")
for p in required:
    print(f"  OK {p} ({p.stat().st_size} bytes)")


## 5. Unit tests


In [ ]:
# 7. Unit tests (no backbone)
from __future__ import annotations
import os
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "qwen_env" / "bin" / "python"
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["RELJOIN_ROOT"] = "/content"
cmd = [str(PYTHON), "-u", str(ROOT / "run_reljoin.py"), "--run-mode", "unit", "--root", "/content"]
log_path = ROOT / "outputs_reljoin_v6_qwen_multiseed" / "logs" / "unit.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
print("$", " ".join(cmd))
with log_path.open("w", encoding="utf-8") as logf:
    proc = subprocess.Popen(
        cmd, cwd="/content", env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
        logf.flush()
    proc.wait()
if proc.returncode != 0:
    tail = "\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-50:])
    raise RuntimeError(f"unit tests exited {proc.returncode}. LOG={log_path}\n\n{tail}")
print("UNIT OK", log_path)


## 6. Download Qwen2.5-VL-7B


In [ ]:
# 5. Download Qwen2.5-VL-7B. PerceptionTest is fetched inside training.
from __future__ import annotations
import os
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "qwen_env" / "bin" / "python"
assert PYTHON.exists(), "Run the qwen_env cell first."
os.environ["PATH"] = f"{ROOT / 'qwen_env' / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
MODEL_DIR = ROOT / "checkpoints" / "qwen25vl_7b"
MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
if not (MODEL_DIR / "config.json").exists():
    print("Downloading Qwen2.5-VL-7B (~18 GB bf16)...")
    py = (
        "import os\n"
        "os.environ['HF_HUB_ENABLE_HF_TRANSFER']='1'\n"
        "from huggingface_hub import snapshot_download\n"
        "snapshot_download(repo_id='Qwen/Qwen2.5-VL-7B-Instruct', "
        "revision='cc594898137f460bfe9f0759e9844b3ce807cfb5', "
        f"local_dir=r'{MODEL_DIR}', "
        "ignore_patterns=['*.msgpack','*.h5','flax_model*','tf_model*'])\n"
        "print('model OK')\n"
    )
    subprocess.run([str(PYTHON), "-c", py], check=True)
assert (MODEL_DIR / "config.json").exists()
print("model OK", MODEL_DIR)


## 7. Train three variants × three seeds


In [ ]:
# 8. Train GlobalTimeFix, HybridNoGlobal, Hybrid for seeds 1, 100, 2027
from __future__ import annotations
import json
import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path("/content")
PYTHON = ROOT / "qwen_env" / "bin" / "python"
OUT_ROOT = ROOT / "outputs_reljoin_v6_qwen_multiseed"
try:
    SEEDS
except NameError:
    SEEDS = [1, 100, 2027]
try:
    SPLIT_SEED
except NameError:
    SPLIT_SEED = 42
try:
    VARIANTS
except NameError:
    VARIANTS = ["GlobalTimeFix", "HybridNoGlobal", "Hybrid"]
try:
    PERSIST
except NameError:
    PERSIST = Path("/content/drive/MyDrive/reljoin_v6_qwen_multiseed_outputs")

LABEL = {
    "GlobalTimeFix": "global-only (local branch off)",
    "HybridNoGlobal": "local-only (global branch off)",
    "Hybrid": "full hybrid",
}
reference_ids = None

def stream(cmd, log_path: Path, env: dict) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("$", " ".join(cmd))
    with log_path.open("w", encoding="utf-8") as logf:
        proc = subprocess.Popen(
            cmd, cwd="/content", env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            logf.write(line)
            logf.flush()
        proc.wait()
    if proc.returncode != 0:
        tail = "\n".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-40:])
        raise RuntimeError(f"exit {proc.returncode}\nLOG={log_path}\n\n{tail}")

for seed in SEEDS:
    seed_dir = OUT_ROOT / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    drive_seed = PERSIST / f"seed_{seed}"
    for variant in VARIANTS:
        ckpt = seed_dir / "checkpoints" / "best_external_val" / f"{variant}.pt"
        print(f"\n=== seed {seed} {variant} ({LABEL[variant]}) ===")
        if ckpt.is_file():
            print(f"skip, checkpoint exists: {ckpt}")
        else:
            env = os.environ.copy()
            env["PYTHONUNBUFFERED"] = "1"
            env["RELJOIN_ROOT"] = "/content"
            env["RELJOIN_PERSIST"] = str(drive_seed)
            cmd = [
                str(PYTHON), "-u", str(ROOT / "run_reljoin.py"),
                "--run-mode", "lite",
                "--root", "/content",
                "--variant", variant,
                "--stage", "joint",
                "--seed", str(seed),
                "--split-seed", str(SPLIT_SEED),
                "--out-dir", str(seed_dir),
            ]
            stream(cmd, OUT_ROOT / "logs" / f"train_seed{seed}_{variant}.log", env)
        assert ckpt.is_file(), f"missing {ckpt}"
        cfg = json.loads((seed_dir / "config.json").read_text())
        assert int(cfg["seed"]) == int(seed), cfg["seed"]
        assert int(cfg["split_seed"]) == int(SPLIT_SEED), cfg["split_seed"]
        manifest = json.loads((seed_dir / "data" / "train_manifest.json").read_text())
        val = json.loads((seed_dir / "data" / "val_manifest.json").read_text())
        ids = manifest["sample_ids"]
        if reference_ids is None:
            reference_ids = ids
            print(f"canonical split train={len(ids)} val={val['n']} videos={len(manifest['video_ids'])}")
        else:
            assert ids == reference_ids, f"seed {seed} train split drifted from split_seed {SPLIT_SEED}"
        print(f"ok {variant}.pt  train={manifest['n']} val={val['n']} split_seed={cfg['split_seed']}")
    if PERSIST.exists() and drive_seed.resolve() != seed_dir.resolve():
        print(f"copy {seed_dir} -> {drive_seed}")
        shutil.copytree(seed_dir, drive_seed, dirs_exist_ok=True)

assert reference_ids, "no manifest written"
print("all seeds finished", SEEDS)


## 8. Loss curves


In [ ]:
# 9. Validation NLL curves
from __future__ import annotations
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

OUT_ROOT = Path("/content/outputs_reljoin_v6_qwen_multiseed")
fig, ax = plt.subplots(figsize=(10, 6))
found = 0
for csv_path in sorted(OUT_ROOT.glob("seed_*/logs/**/train_metrics.csv")):
    df = pd.read_csv(csv_path)
    if df.empty or "val_nll" not in df.columns:
        continue
    seed = csv_path.parts[csv_path.parts.index("logs") - 1]
    name = str(df["run_name"].iloc[0]) if "run_name" in df.columns else csv_path.parent.name
    ax.plot(df["epoch"], df["val_nll"], marker="o", label=f"{seed} {name}")
    found += 1
ax.set_xlabel("epoch")
ax.set_ylabel("val NLL")
ax.set_title("Qwen multi-seed validation NLL")
if found:
    ax.legend(fontsize=8)
    fig.tight_layout()
    out = OUT_ROOT / "loss_curves.png"
    fig.savefig(out, dpi=120)
    print("wrote", out, "curves", found)
else:
    print("no train_metrics.csv yet")
plt.show()


## 9. Export


In [ ]:
# 10. Copy seed folders to Drive and zip
from __future__ import annotations
import shutil
from pathlib import Path

OUT_ROOT = Path("/content/outputs_reljoin_v6_qwen_multiseed")
try:
    PERSIST
except NameError:
    PERSIST = Path("/content/drive/MyDrive/reljoin_v6_qwen_multiseed_outputs")
try:
    SEEDS
except NameError:
    SEEDS = [1, 100, 2027]

if PERSIST.exists():
    for seed in SEEDS:
        src = OUT_ROOT / f"seed_{seed}"
        if src.exists() and src.resolve() != (PERSIST / f"seed_{seed}").resolve():
            shutil.copytree(src, PERSIST / f"seed_{seed}", dirs_exist_ok=True)
            print("copied", src)
    curves = OUT_ROOT / "loss_curves.png"
    if curves.is_file():
        shutil.copy2(curves, PERSIST / curves.name)
zip_path = Path("/content/qwen_multiseed_train")
shutil.make_archive(str(zip_path), "zip", OUT_ROOT)
print("zip", zip_path.with_suffix(".zip"))
try:
    from google.colab import files
    files.download(str(zip_path.with_suffix(".zip")))
except Exception as exc:
    print("download skipped:", exc)
